# ELEC 475 Lab 2 — Colab experiment runner
Use **Runtime → Change runtime type → T4 GPU** before running the notebook. The cells are split into phases so outputs can be saved between Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

ZIP_PATH = Path('/content/drive/MyDrive/ELEC475_Lab2_code.zip')
PROJECT_DIR = Path('/content/lab2_template')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/ELEC475_Lab2_completed_outputs')

assert ZIP_PATH.exists(), f'Project archive not found: {ZIP_PATH}'
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.unpack_archive(ZIP_PATH, '/content')
assert (PROJECT_DIR / 'train.py').exists()
%cd /content/lab2_template

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable a GPU runtime before training.'
!python -m pytest -q

## Phase 1 — in-lab prerequisites
Trains the base and modernized LeNet models for five epochs without augmentation.

In [ ]:
!python run_experiments.py --phase inlab --device cuda --seeds 42 --skip-profile

## Phase 2 — CIFAR-10 post-lab experiments
Runs no-normalization, LRN, BatchNorm, augmentation, and Model Zoo baselines.

In [ ]:
!python run_experiments.py --phase cifar --device cuda --seeds 42 --skip-profile

## Phase 3 — required Tiny ImageNet AlexNet run and profiling

In [ ]:
!python run_experiments.py --phase tiny --device cuda --seeds 42

## Save outputs back to Drive
Run this after each phase so checkpoints, figures, and JSON metrics survive a Colab disconnect.

In [ ]:
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
for folder in ('checkpoints', 'plots', 'results'):
    source = PROJECT_DIR / folder
    destination = DRIVE_OUTPUT / folder
    if source.exists():
        shutil.copytree(source, destination, dirs_exist_ok=True)
print('Saved outputs to', DRIVE_OUTPUT)

## Optional — three-seed statistics
The report asks for mean ± standard deviation. Run this cell if time permits; completed seed-42 outputs are skipped automatically.

In [ ]:
!python run_experiments.py --phase cifar --device cuda --seeds 42 123 2026 --skip-profile

## Optional — populate every Tiny ImageNet cell in Table 7.3b
This is substantially more GPU-intensive than the explicitly requested AlexNet Tiny ImageNet run.

In [ ]:
!python run_experiments.py --phase full --device cuda --seeds 42 --skip-profile